In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify credentials are set
assert os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'), (
    "Kaggle credentials not found!\n")
print(f"Kaggle credentials loaded for: {os.getenv('KAGGLE_USERNAME')}")

Kaggle credentials loaded for: your_kaggle_username_here


In [8]:
import kagglehub

# Downloads dataset to local cache (~/.cache/kagglehub/...) on first run
# Subsequent runs use the cached copy - no re-download needed
path = kagglehub.dataset_download("davidcariboo/player-scores")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Ehtesham-PC\.cache\kagglehub\datasets\davidcariboo\player-scores\versions\671


In [4]:
# Checking which files downloaded

import os
print(os.listdir(path))

['appearances.csv', 'clubs.csv', 'club_games.csv', 'competitions.csv', 'countries.csv', 'games.csv', 'game_events.csv', 'game_lineups.csv', 'national_teams.csv', 'players.csv', 'player_valuations.csv', 'transfers.csv']


In [5]:
players = pd.read_csv(f"{path}/players.csv")
valuations = pd.read_csv(f"{path}/player_valuations.csv")
appearances = pd.read_csv(f"{path}/appearances.csv")

print("players shape:", players.shape)
print("valuations shape:", valuations.shape)
print("appearances shape:", appearances.shape)

players shape: (48380, 26)
valuations shape: (656301, 6)
appearances shape: (1889406, 13)


In [6]:
# Players

print(players.columns.tolist())
print(players.head())

['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']
   player_id first_name     last_name                name  last_season  \
0         10   Miroslav         Klose      Miroslav Klose         2015   
1         26      Roman  Weidenfeller  Roman Weidenfeller         2017   
2         65    Dimitar      Berbatov    Dimitar Berbatov         2015   
3         77        NaN         Lúcio               Lúcio         2012   
4         80        Tom        Starke          Tom Starke         2017   

   current_club_id         player_code    country_of_birth city_of_birth  \

In [7]:
# Valuations

print(valuations.columns.tolist())
print(valuations.head())

['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']
   player_id        date  market_value_in_eur current_club_name  \
0     405973  2000-01-20               150000           Unknown   
1     342216  2001-07-20               100000           Unknown   
2       3132  2003-12-09               400000       Dynamo Kyiv   
3       6893  2003-12-15               900000       Galatasaray   
4         10  2004-10-04              7000000  SV Werder Bremen   

   current_club_id player_club_domestic_competition_id  
0           3057.0                                 BE1  
1           1241.0                                 SC1  
2            126.0                                 TR1  
3            984.0                                 GB1  
4            398.0                                 IT1  


In [48]:
# Appearances

print(appearances.columns.tolist())
print(appearances.head())

['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']
    appearance_id  game_id  player_id  player_club_id  player_current_club_id  \
0   2231978_38004  2231978      38004             853                     235   
1   2233748_79232  2233748      79232            8841                    2698   
2   2234413_42792  2234413      42792            6251                     465   
3   2234418_73333  2234418      73333            1274                      76   
4  2234421_122011  2234421     122011             195                    3008   

         date       player_name competition_id  yellow_cards  red_cards  \
0  2012-07-03  Aurélien Joachim            CLQ             0          0   
1  2012-07-05    Ruslan Abyshov            ELQ             0          0   
2  2012-07-05       Sander Puri            ELQ             0          0   
3  2012-07-05   Veg

In [49]:
critical_cols = ['date_of_birth', 'position', 'player_id', 'height_in_cm', 'foot']
print(players[critical_cols].isnull().sum())
print(f"\nTotal players: {len(players)}")

date_of_birth      49
position            0
player_id           0
height_in_cm     3783
foot             5288
dtype: int64

Total players: 48380


In [50]:
players = players.dropna(subset=['date_of_birth'])
players['date_of_birth'] = pd.to_datetime(players['date_of_birth'])
print(f"Players remaining: {len(players)}")

Players remaining: 48331


In [51]:
valuations['date'] = pd.to_datetime(valuations['date'])
print(valuations.dtypes)

player_id                                       int64
date                                   datetime64[ns]
market_value_in_eur                             int64
current_club_name                              object
current_club_id                               float64
player_club_domestic_competition_id            object
dtype: object


In [52]:
valuations = valuations.merge(
    players[['player_id', 'date_of_birth', 'position', 'foot', 'height_in_cm', 'international_caps', 'country_of_citizenship']],
    on='player_id',
    how='left'
)

valuations['age_at_valuation'] = (
    (valuations['date'] - valuations['date_of_birth']).dt.days / 365.25
)

# filtering out invalid ages
valuations = valuations[
    (valuations['age_at_valuation'] >= 15) &
    (valuations['age_at_valuation'] <= 40)
]

print(f"Valuation records after age filter: {len(valuations)}")
print(valuations[['player_id', 'date', 'age_at_valuation', 'market_value_in_eur']].head(20))

Valuation records after age filter: 655058
    player_id       date  age_at_valuation  market_value_in_eur
2        3132 2003-12-09         23.748118               400000
3        6893 2003-12-15         20.098563               900000
4          10 2004-10-04         26.321697              7000000
5          26 2004-10-04         24.161533              1500000
6          65 2004-10-04         23.676934              8000000
7          77 2004-10-04         26.409309             13000000
8          80 2004-10-04         23.548255               400000
9         109 2004-10-04         26.464066              9500000
10        123 2004-10-04         23.912389              9500000
11        132 2004-10-04         24.000000             13000000
12        162 2004-10-04         28.309377              1250000
13        215 2004-10-04         23.134839              7500000
14        264 2004-10-04         23.556468               250000
15        276 2004-10-04         22.472279               2500

In [53]:
# filter to valuations where player was under 21
young_valuations = valuations[valuations['age_at_valuation'] <= 21].copy()

# get the earliest valuation for each player under 21
first_valuation = young_valuations.sort_values('date').groupby('player_id').first().reset_index()

print(f"Players with at least one valuation under 21: {len(first_valuation)}")
print(first_valuation[['player_id', 'date', 'age_at_valuation', 'market_value_in_eur']].head())

Players with at least one valuation under 21: 30757
   player_id       date  age_at_valuation  market_value_in_eur
0       1428 2004-10-04         20.914442               900000
1       1606 2004-10-04         19.526352               150000
2       1716 2004-10-04         20.618754              1500000
3       1784 2004-10-04         19.718001               200000
4       2050 2004-10-04         20.851472              2000000


In [54]:
# filter to valuations between age 21 and 25 (3 year window after the under 21 period)
future_valuations = valuations[
    (valuations['age_at_valuation'] > 21) &
    (valuations['age_at_valuation'] <= 25)
].copy()

# get peak value in that window
peak_valuation = future_valuations.groupby('player_id')['market_value_in_eur'].max().reset_index()
peak_valuation.columns = ['player_id', 'peak_value_21_25']

print(f"Players with valuation between 21-25: {len(peak_valuation)}")
print(peak_valuation.head())

Players with valuation between 21-25: 34589
   player_id  peak_value_21_25
0         26           5000000
1         65          14500000
2         80            400000
3        123           9500000
4        132          13000000


In [55]:
label_df = first_valuation.merge(peak_valuation, on='player_id', how='inner')
label_df = label_df.rename(columns={'market_value_in_eur': 'starting_value'})

print(f"Players we can label: {len(label_df)}")
print(label_df[['player_id', 'starting_value', 'peak_value_21_25']].head(20))

Players we can label: 25621
    player_id  starting_value  peak_value_21_25
0        1428          900000           5000000
1        1606          150000            450000
2        1716         1500000           4500000
3        1784          200000          14000000
4        2050         2000000           5000000
5        2148          200000           1300000
6        2219         4500000          21000000
7        2233          500000           4000000
8        2358          200000           4000000
9        2421          500000           4500000
10       2514         3250000          16000000
11       2623          800000           8000000
12       2712         1000000           1750000
13       2857          100000           1500000
14       2865         1250000           5000000
15       2923         3000000           5500000
16       2926          500000           2100000
17       2989          500000           5000000
18       2998         2250000           8900000
19       317

In [56]:
def is_wonderkid(row):
    start = row['starting_value']
    peak = row['peak_value_21_25']

    if start < 1_000_000:
        return 1 if peak >= 5_000_000 else 0

    elif start < 5_000_000:
        return 1 if peak >= start * 3 else 0

    elif start < 15_000_000:
        return 1 if peak >= start * 2.5 else 0

    else:
        return 1 if peak >= start * 2 else 0

label_df['wonderkid'] = label_df.apply(is_wonderkid, axis=1)

print(label_df['wonderkid'].value_counts())
print(f"\nWonderkid %: {label_df['wonderkid'].mean() * 100:.1f}%")

wonderkid
0    21751
1     3870
Name: count, dtype: int64

Wonderkid %: 15.1%


In [57]:
print(appearances.columns.tolist())
print(appearances[['player_id', 'date', 'goals', 'assists', 'minutes_played', 'yellow_cards', 'red_cards']].head())
print(f"\nDate range: {appearances['date'].min()} to {appearances['date'].max()}")

['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']
   player_id        date  goals  assists  minutes_played  yellow_cards  \
0      38004  2012-07-03      2        0              90             0   
1      79232  2012-07-05      0        0              90             0   
2      42792  2012-07-05      0        0              45             0   
3      73333  2012-07-05      0        0              90             0   
4     122011  2012-07-05      0        1              90             0   

   red_cards  
0          0  
1          0  
2          0  
3          0  
4          0  

Date range: 2012-07-03 to 2026-06-28


In [58]:
appearances['date'] = pd.to_datetime(appearances['date'])

# get each player's anchor date (their first valuation date, under 21) from first_valuation
anchor_dates = first_valuation[['player_id', 'date']].rename(columns={'date': 'anchor_date'})

appearances_anchored = appearances.merge(anchor_dates, on='player_id', how='inner')
print(f"Appearance records with an anchor date: {len(appearances_anchored)}")
print(appearances_anchored[['player_id', 'date', 'anchor_date']].head())

Appearance records with an anchor date: 1483632
   player_id       date anchor_date
0      38004 2012-07-03  2006-02-15
1      42792 2012-07-05  2008-07-05
2      73333 2012-07-05  2008-10-13
3     122011 2012-07-05  2009-08-30
4     146889 2012-07-05  2010-04-08


In [59]:
appearances_anchored['days_from_anchor'] = (appearances_anchored['date'] - appearances_anchored['anchor_date']).dt.days

early_career = appearances_anchored[
    (appearances_anchored['days_from_anchor'] >= -180) &  # allow a bit before valuation too
    (appearances_anchored['days_from_anchor'] <= 730)     # up to 2 years after
].copy()

print(f"Early career appearance records: {len(early_career)}")
print(f"Unique players with early career appearances: {early_career['player_id'].nunique()}")

Early career appearance records: 129232
Unique players with early career appearances: 9132


In [60]:
player_features = early_career.groupby('player_id').agg(
    total_games=('appearance_id', 'count'),
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    total_yellow=('yellow_cards', 'sum'),
    total_red=('red_cards', 'sum')
).reset_index()

# per-game rates
player_features['goals_per_game'] = player_features['total_goals'] / player_features['total_games']
player_features['assists_per_game'] = player_features['total_assists'] / player_features['total_games']
player_features['minutes_per_game'] = player_features['total_minutes'] / player_features['total_games']
player_features['cards_per_game'] = (player_features['total_yellow'] + player_features['total_red']) / player_features['total_games']

print(player_features.shape)
print(player_features.head())

(9132, 11)
   player_id  total_games  total_goals  total_assists  total_minutes  \
0       5336           36            0              1           1996   
1       7161            3            0              1            170   
2      20463           10            3              0            643   
3      21297           15            0              0            518   
4      22686            3            0              0             33   

   total_yellow  total_red  goals_per_game  assists_per_game  \
0             3          0             0.0          0.027778   
1             0          0             0.0          0.333333   
2             1          0             0.3          0.000000   
3             0          0             0.0          0.000000   
4             0          0             0.0          0.000000   

   minutes_per_game  cards_per_game  
0         55.444444        0.083333  
1         56.666667        0.000000  
2         64.300000        0.100000  
3         34.533333

In [61]:
final_df = label_df.reset_index().merge(player_features, on='player_id', how='left')

print(f"Total players: {len(final_df)}")
print(f"Players with appearance data: {final_df['total_games'].notna().sum()}")
print(f"Players missing appearance data: {final_df['total_games'].isna().sum()}")

Total players: 25621
Players with appearance data: 7297
Players missing appearance data: 18324


In [62]:
# players with no appearance data likely means: no top-tier minutes recorded early on
# fill numeric performance stats with 0 rather than dropping them - "no data" is itself informative
perf_cols = ['total_games', 'total_goals', 'total_assists', 'total_minutes',
             'total_yellow', 'total_red', 'goals_per_game', 'assists_per_game',
             'minutes_per_game', 'cards_per_game']

final_df[perf_cols] = final_df[perf_cols].fillna(0)

print(final_df[perf_cols].isnull().sum())
print(f"\nFinal shape: {final_df.shape}")

total_games         0
total_goals         0
total_assists       0
total_minutes       0
total_yellow        0
total_red           0
goals_per_game      0
assists_per_game    0
minutes_per_game    0
cards_per_game      0
dtype: int64

Final shape: (25621, 26)


In [63]:
# print(first_valuation.columns.tolist())
# print(first_valuation.index.name)

In [64]:
print(final_df.columns.tolist())
print(final_df.dtypes)
print(final_df['wonderkid'].value_counts())

['index', 'player_id', 'date', 'starting_value', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id', 'date_of_birth', 'position', 'foot', 'height_in_cm', 'international_caps', 'country_of_citizenship', 'age_at_valuation', 'peak_value_21_25', 'wonderkid', 'total_games', 'total_goals', 'total_assists', 'total_minutes', 'total_yellow', 'total_red', 'goals_per_game', 'assists_per_game', 'minutes_per_game', 'cards_per_game']
index                                           int64
player_id                                       int64
date                                   datetime64[ns]
starting_value                                  int64
current_club_name                              object
current_club_id                               float64
player_club_domestic_competition_id            object
date_of_birth                          datetime64[ns]
position                                       object
foot                                           object
height_in

In [65]:
# columns that leak the label or are just identifiers/metadata - exclude from features
feature_cols = ['position', 'foot', 'height_in_cm', 'international_caps','total_games', 'goals_per_game', 'assists_per_game','minutes_per_game', 'cards_per_game']

model_df = final_df[feature_cols + ['wonderkid']].copy()
print(model_df.shape)
print(model_df.isnull().sum())
print(model_df.dtypes)

(25621, 10)
position                  0
foot                   1131
height_in_cm            678
international_caps    14429
total_games               0
goals_per_game            0
assists_per_game          0
minutes_per_game          0
cards_per_game            0
wonderkid                 0
dtype: int64
position               object
foot                   object
height_in_cm          float64
international_caps    float64
total_games           float64
goals_per_game        float64
assists_per_game      float64
minutes_per_game      float64
cards_per_game        float64
wonderkid               int64
dtype: object


In [66]:
print(model_df['foot'].value_counts())
print(model_df['height_in_cm'].describe())

foot
right    17285
left      6161
both      1044
Name: count, dtype: int64
count    24943.000000
mean       182.113539
std          7.209380
min         17.000000
25%        177.000000
50%        182.000000
75%        187.000000
max        210.000000
Name: height_in_cm, dtype: float64


In [67]:
# check unrealistic height values first
print((model_df['height_in_cm'] < 140).sum())
print(model_df[model_df['height_in_cm'] < 140]['height_in_cm'].describe())

# drop international_caps - unreliable
model_df = model_df.drop('international_caps', axis=1)

# treat unrealistic heights as missing, then impute
model_df.loc[model_df['height_in_cm'] < 140, 'height_in_cm'] = np.nan
model_df['foot'] = model_df['foot'].fillna(model_df['foot'].mode()[0])
model_df['height_in_cm'] = model_df['height_in_cm'].fillna(model_df['height_in_cm'].median())

print(model_df.isnull().sum())
print(model_df.shape)

5
count     5.000000
mean     17.800000
std       0.447214
min      17.000000
25%      18.000000
50%      18.000000
75%      18.000000
max      18.000000
Name: height_in_cm, dtype: float64
position            0
foot                0
height_in_cm        0
total_games         0
goals_per_game      0
assists_per_game    0
minutes_per_game    0
cards_per_game      0
wonderkid           0
dtype: int64
(25621, 9)


In [68]:
from sklearn.model_selection import train_test_split

X = model_df.drop('wonderkid', axis=1)
y = model_df['wonderkid']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train wonderkid %: {y_train.mean()*100:.1f}%")
print(f"Test wonderkid %: {y_test.mean()*100:.1f}%")

Train shape: (20496, 8)
Test shape: (5125, 8)
Train wonderkid %: 15.1%
Test wonderkid %: 15.1%


In [69]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_cols = ['position', 'foot']
numerical_cols = ['height_in_cm', 'total_games', 'goals_per_game','assists_per_game', 'minutes_per_game', 'cards_per_game']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# quick sanity check - fit_transform on train only
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"Transformed train shape: {X_train_transformed.shape}")

Transformed train shape: (20496, 14)


In [70]:
# Decision Tree Classifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])

dt_pipeline.fit(X_train, y_train)
dt_preds = dt_pipeline.predict(X_test)

print("Decision Tree")
print(classification_report(y_test, dt_preds))
print(confusion_matrix(y_test, dt_preds))

Decision Tree
              precision    recall  f1-score   support

           0       0.86      0.79      0.83      4351
           1       0.20      0.30      0.24       774

    accuracy                           0.72      5125
   macro avg       0.53      0.55      0.53      5125
weighted avg       0.76      0.72      0.74      5125

[[3448  903]
 [ 543  231]]


In [71]:
#Knn

from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)
knn_preds = knn_pipeline.predict(X_test)

print("kNN")
print(classification_report(y_test, knn_preds))
print(confusion_matrix(y_test, knn_preds))

kNN
              precision    recall  f1-score   support

           0       0.87      0.98      0.92      4351
           1       0.56      0.16      0.25       774

    accuracy                           0.85      5125
   macro avg       0.71      0.57      0.58      5125
weighted avg       0.82      0.85      0.82      5125

[[4251  100]
 [ 649  125]]


In [72]:
# Naiive Bayes

from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer

nb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('to_dense', FunctionTransformer(lambda x: x.toarray() if hasattr(x, 'toarray') else x)),
    ('classifier', GaussianNB())
])

nb_pipeline.fit(X_train, y_train)
nb_preds = nb_pipeline.predict(X_test)

print("Naive Bayes")
print(classification_report(y_test, nb_preds))
print(confusion_matrix(y_test, nb_preds))

Naive Bayes
              precision    recall  f1-score   support

           0       0.94      0.13      0.23      4351
           1       0.16      0.95      0.28       774

    accuracy                           0.25      5125
   macro avg       0.55      0.54      0.25      5125
weighted avg       0.82      0.25      0.24      5125

[[ 568 3783]
 [  38  736]]


In [73]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__min_samples_leaf': [1, 5, 10, 20],
    'classifier__criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    dt_pipeline,
    param_grid,
    scoring='f1',  # optimizing F1 on the positive (wonderkid) class
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.3f}")

best_dt = grid_search.best_estimator_
best_dt_preds = best_dt.predict(X_test)

print("\n=== Tuned Decision Tree ===")
print(classification_report(y_test, best_dt_preds))
print(confusion_matrix(y_test, best_dt_preds))

Best params: {'classifier__criterion': 'entropy', 'classifier__max_depth': 3, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2}
Best CV F1 score: 0.350

=== Tuned Decision Tree ===
              precision    recall  f1-score   support

           0       0.89      0.85      0.87      4351
           1       0.34      0.44      0.38       774

    accuracy                           0.78      5125
   macro avg       0.61      0.64      0.62      5125
weighted avg       0.81      0.78      0.80      5125

[[3677  674]
 [ 434  340]]
